[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Cosmopedia_TriHead_Transformer.ipynb)

# 🧠 CerebroSpheroid V5: Cosmopedia ➔ Tri-Head Reasoning Dataset Transformer
### Transform Static Educational Textbooks into 3-Tier Cognitive Emission (`<think>`, `<act>`, `<answer>`) via Gemini API

**CerebroSpheroid V5 (Doty)** replaces passive unsupervised document memorization with **3-Tier Active Cognitive Emission**:
1. **💭 Deliberation (`<think> ... </think>`)**: Subconscious multi-step scratchpad for hypothesis formulation, derivation, and self-reflection.
2. **⚡ Grounded Action (`<act> ... </act>`)**: Non-conversational tool execution: `calc(...)` arithmetic, `sql: SELECT ...` database fact lookup, `python: ...` code synthesis, and `mermaid: ...` diagrams.
3. **🗣️ Conversational Answer**: Clear, engaging speech grounded directly in the reasoning and tool observations.

---
### 🎯 Purpose of this Notebook:
Raw **Cosmopedia** (`HuggingFaceTB/cosmopedia`) contains rich educational content (math, openstax textbooks, science, stories), but it is written as passive, monologic articles. This notebook streams passages from Cosmopedia and uses **Gemini 2.5 Flash** to convert each passage into an active **Tri-Head SFT record** (`prompt <= 512 tok`, `response <= 1024 tok`, `temporal_gate: [1, 0, 1]` or `[1, 1, 1]`).


## 📦 Step 1: Install Prerequisites & Setup Gemini API Key
Installs the official Google GenAI SDK (`google-genai`), Hugging Face `datasets`, and sets up your Gemini API key (via Colab Secrets or interactive input).


In [ ]:
# Install latest Google GenAI SDK and Hugging Face streaming datasets
!pip install -q -U google-genai datasets huggingface_hub tqdm pydantic

import os, sys, getpass

# Retrieve Gemini API Key from Colab Secrets or prompt
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    print("🔑 Enter your Gemini API Key (get one free at https://aistudio.google.com/):")
    GEMINI_API_KEY = getpass.getpass("Gemini API Key: ").strip()

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Gemini API client successfully initialized!")


## ⚙️ Step 2: Configure Transformation Parameters
Choose your Cosmopedia subset and target dataset size:
- `auto_math_text`: Mathematics, algebra, calculus, and quantitative problems (generates `<act> calc(...) </act>`).
- `khanacademy`: Elementary and high-school conceptual science, humanities, and history (generates SQL / dialogue).
- `openstax`: College-level physics, chemistry, biology, and economics.
- `stanford`: Advanced computer science, algorithms, and logic (generates `<act> python: ... </act>`).
- `stories` / `wikihow`: Practical real-world problem-solving, workflows, and procedures.


In [ ]:
# ==============================================================================
# ⚙️ Pipeline Configuration
# ==============================================================================
GEMINI_MODEL       = "gemini-2.5-flash"    # Fast, high-capacity, low-latency reasoning
COSMOPEDIA_SUBSET  = "auto_math_text"     # auto_math_text, openstax, khanacademy, stanford
TARGET_SAMPLES     = 500                  # Number of Tri-Head SFT records to generate
OUTPUT_FILE        = "/content/cosmopedia_trihead_sft.jsonl"
MAX_WORKERS        = 5                    # Parallel transformation threads
MAX_PROMPT_LEN     = 512                  # Token budget for sensory stimulus (<stim>)
MAX_RESPONSE_LEN   = 1024                 # Token budget for response (<think> + <act> + answer)

print(f"Target Configuration:")
print(f"  • Model     : {GEMINI_MODEL}")
print(f"  • Subset    : {COSMOPEDIA_SUBSET}")
print(f"  • Target    : {TARGET_SAMPLES:,} samples")
print(f"  • Output    : {OUTPUT_FILE}")
print(f"  • Workers   : {MAX_WORKERS}")


## 🧬 Step 3: Define Tri-Head System Instruction & Parser
The transformation engine instructs Gemini to convert passive educational text into active Tri-Head cognitive format:
1. **Prompt**: Natural user inquiry asking about the underlying concept.
2. **Thought**: Private deliberation scratchpad inside `<think> ... </think>`.
3. **Action**: Tool execution inside `<act> ... </act>` (`calc`, `sql`, `python`, `mermaid`) when quantitative or relational.
4. **Answer**: Direct pedagogical answer grounded in the deliberation.
5. **Gate**: `[1, 1, 1]` if tool action is present, `[1, 0, 1]` otherwise.


In [ ]:
import json, re, time, random
from google.genai import types
from datasets import load_dataset

SYSTEM_INSTRUCTION = """You are an expert cognitive curriculum engineer for CerebroSpheroid V5, an advanced tri-head reasoning AI.
Your job is to transform raw, passive educational passages from textbooks (Cosmopedia) into high-density, interactive TRI-HEAD training examples.

A Tri-Head example consists of:
1. `prompt`: A natural, curious user inquiry or problem inspired by the concepts in the text (never "Summarize this").
2. `thought`: 2-4 lines of explicit internal cognitive deliberation. Dissect intent, plan the steps, state hypotheses or formulas, and anticipate edge cases.
3. `action`: Optional grounded tool call if beneficial:
   - For arithmetic/equations: `calc(<math expression>)`
   - For relational facts: `sql: SELECT ...` or `db_lookup("Entity", "Attribute")`
   - For programming: `python: ...`
   - For diagrams/flows: `mermaid: graph TD ...`
   - Leave empty ("") if purely conceptual dialogue.
4. `answer`: A clear, direct, pedagogical answer grounded in the deliberation and tool action.
5. `category`: One of: "conversational_dialogue", "reasoning_math", "db_fact_retrieval", "python_code", "agentic_modalities"

Format strictly as JSON with keys: "prompt", "thought", "action", "answer", "category"
"""

def call_gemini_transform(client, text_passage, model_name="gemini-2.5-flash", max_retries=5):
    prompt = f"Transform the following educational passage into a high-quality Tri-Head reasoning example:\n\n--- PASSAGE ---\n{text_passage.strip()[:2000]}\n--- END PASSAGE ---\n"
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    response_mime_type="application/json",
                    temperature=0.7,
                ),
            )
            raw_text = response.text.strip()
            if raw_text.startswith("```json"):
                raw_text = raw_text[7:]
            if raw_text.endswith("```"):
                raw_text = raw_text[:-3]
            data = json.loads(raw_text.strip())
            if isinstance(data, list) and data:
                data = data[0]
            if "prompt" in data and "thought" in data and "answer" in data:
                return data
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "quota" in err_str or "rate" in err_str:
                time.sleep((2 ** attempt) + random.uniform(0.5, 1.5))
            else:
                time.sleep(1.0)
    return None

def format_trihead_record(raw_result):
    prompt_raw = raw_result.get("prompt", "").strip()
    thought_raw = raw_result.get("thought", "").strip()
    action_raw = raw_result.get("action", "").strip()
    answer_raw = raw_result.get("answer", "").strip()
    category = raw_result.get("category", "conversational_dialogue").strip()

    prompt_formatted = f"User: {prompt_raw}" if not prompt_raw.lower().startswith("user:") else prompt_raw

    response_blocks = []
    if thought_raw:
        clean_thought = thought_raw.replace("<think>", "").replace("</think>", "").strip()
        response_blocks.append(f"<think>\n{clean_thought}\n</think>")

    has_action = bool(action_raw and action_raw.lower() not in ("none", "null", '""', "n/a"))
    if has_action:
        clean_action = action_raw.replace("<act>", "").replace("</act>", "").strip()
        response_blocks.append(f"<act>\n{clean_action}\n</act>")

    clean_answer = answer_raw.replace("<answer>", "").replace("</answer>", "").strip()
    response_blocks.append(clean_answer)

    full_response = "\n".join(response_blocks)
    temporal_gate = [1, 1, 1] if has_action else [1, 0, 1]

    return {
        "prompt": prompt_formatted,
        "response": full_response,
        "category": category,
        "temporal_gate": temporal_gate,
    }


## 🚀 Step 4: Stream Cosmopedia & Run Concurrent Transformation
Streams passages on-the-fly and processes them in parallel across worker threads with live progress logging.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from IPython.display import display, Markdown

print(f"🌐 Connecting to HuggingFaceTB/cosmopedia ({COSMOPEDIA_SUBSET})...")
dataset_stream = load_dataset("HuggingFaceTB/cosmopedia", COSMOPEDIA_SUBSET, split="train", streaming=True)

passages = []
print("⏳ Fetching candidate passages...")
for item in dataset_stream:
    text = (item.get("text", "") or item.get("content", "")).strip()
    if len(text) >= 150:
        passages.append(text)
        if len(passages) >= TARGET_SAMPLES:
            break
print(f"✓ Retrieved {len(passages):,} passages from Cosmopedia!\n")

def process_single(passage):
    raw = call_gemini_transform(client, passage, model_name=GEMINI_MODEL)
    if raw:
        return format_trihead_record(raw)
    return None

saved_records = []
with open(OUTPUT_FILE, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_single, p): i for i, p in enumerate(passages)}
        with tqdm(total=len(passages), desc="Transforming Tri-Head Records") as pbar:
            for fut in as_completed(futures):
                rec = fut.result()
                if rec:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    out_f.flush()
                    saved_records.append(rec)
                pbar.update(1)

print(f"\n🎉 Successfully transformed {len(saved_records):,} Tri-Head SFT records!")
print(f"📁 Saved to: {OUTPUT_FILE}")


## 📊 Step 5: Inspect Formatted Records & Merge with Cerebro Dataset
Previews transformed samples, displays gate distributions, and provides direct download or merge commands.


In [ ]:
from collections import Counter

# Display statistics
cat_counts = Counter(r["category"] for r in saved_records)
gate_counts = Counter(str(r["temporal_gate"]) for r in saved_records)

print("=" * 60)
print("📊 TRANSFORMED DATASET SUMMARY:")
print(f"Total Generated Samples: {len(saved_records):,}")
print("\nCategory Breakdown:")
for cat, cnt in cat_counts.most_common():
    print(f"  • {cat:25s}: {cnt:4d} ({cnt/max(1, len(saved_records))*100:.1f}%)")

print("\nTemporal Gate Distribution:")
for g, cnt in gate_counts.most_common():
    label = "[Think + Act + Answer]" if "1, 1, 1" in g else "[Think + Answer]"
    print(f"  • Gate {g} {label:24s}: {cnt:4d} ({cnt/max(1, len(saved_records))*100:.1f}%)")
print("=" * 60)

# Preview first 2 generated samples
print("\n🔍 PREVIEW GENERATED SAMPLES:")
for i, r in enumerate(saved_records[:2]):
    print(f"\n--- [Sample {i+1}] ({r['category']} | Gate: {r['temporal_gate']}) ---")
    print(f"{r['prompt']}\n")
    print(f"{r['response']}\n")

# Optional: Colab file download
try:
    from google.colab import files
    print("\n📥 Ready to download your generated dataset! Run:")
    print("   files.download(OUTPUT_FILE)")
except Exception:
    pass
